#Code for training a CNN (resnet) to classify fruits and vegetables

In [10]:
#Import required packages
import kagglehub
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchprofile import profile_macs

In [11]:
#Hyperparmeters
batch_size = 10
learning_rate = 1e-4
num_epochs = 10
patience = 3 #Number of epochs to wait for improvement before stopping training

In [12]:
#Helper functions (from assignment 1)
def get_model_macs(model, inputs) -> int:
    return profile_macs(model, inputs)

def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements

def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=False) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

In [13]:
# Download latest version
path = kagglehub.dataset_download("sunnyagarwal427444/food-ingredient-dataset-51")

print("Path to dataset files:", path)

for root, dirs, files in os.walk(path):
    print(root)
    break

Path to dataset files: /home/alkelly2/.cache/kagglehub/datasets/sunnyagarwal427444/food-ingredient-dataset-51/versions/1
/home/alkelly2/.cache/kagglehub/datasets/sunnyagarwal427444/food-ingredient-dataset-51/versions/1


In [14]:
#Load the Dataset

train_dataset = datasets.ImageFolder(root=path+"/huggingface/Train")
val_dataset = datasets.ImageFolder(root=path+"/huggingface/val")

print("Number of classes:", len(train_dataset.classes))
print("Class names:", train_dataset.classes)

Number of classes: 51
Class names: ['Amaranth', 'Apple', 'Banana', 'Beetroot', 'Bell pepper', 'Bitter Gourd', 'Blueberry', 'Bottle Gourd', 'Broccoli', 'Cabbage', 'Cantaloupe', 'Capsicum', 'Carrot', 'Cauliflower', 'Chilli pepper', 'Coconut', 'Corn', 'Cucumber', 'Dragon_fruit', 'Eggplant', 'Fig', 'Garlic', 'Ginger', 'Grapes', 'Jalepeno', 'Kiwi', 'Lemon', 'Mango', 'Okra', 'Onion', 'Orange', 'Paprika', 'Pear', 'Peas', 'Pineapple', 'Pomegranate', 'Potato', 'Pumpkin', 'Raddish', 'Raspberry', 'Ridge Gourd', 'Soy beans', 'Spinach', 'Spiny Gourd', 'Sponge Gourd', 'Strawberry', 'Sweetcorn', 'Sweetpotato', 'Tomato', 'Turnip', 'Watermelon']


In [15]:
#Apply transform to the dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # ResNet standard
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean
        std=[0.229, 0.224, 0.225]    # ImageNet std
    )
])

train_dataset.transform = transform #set the transform attribute of the ImageFolder class
val_dataset.transform = transform

#Create dataloaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)

In [16]:
#Load Resnet model
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(train_dataset.classes)) #make the last fc layer output the correct number of classes

#Setup GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/home/alkelly2/ece556/myvenv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/alkelly2/ece556/myvenv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [17]:
#Train the model
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

#setup early exiting
best_loss = float('inf')
epochs_without_improvement = 0

#training loop
for epoch in range(num_epochs):
    print("Staring epoch", epoch)
    model.train() #enable training mode
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images) #compute predictions
        loss = criterion(outputs, labels) #compute loss
        loss.backward() #backward pass
        optimizer.step() #update model weights

        #track training loss
        running_loss += loss.item()

        #track training accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    #calculate epoch loss
    epoch_loss = running_loss / len(train_loader)

    #calculate epoch accuracy
    epoch_acc = 100 * correct / total

    print("Epoch: ", epoch, "/", num_epochs, " Loss: ", epoch_loss, " Accuracy:", epoch_acc)
    
    #check if loss is still improving
    if epoch_loss < best_loss: #loss improved
        best_loss = epoch_loss
        epochs_without_improvement = 0
        
    else: #loss did not improve
        epochs_without_improvement += 1
        
    #stop training if loss is not improving
    if epochs_without_improvement >= patience:
        print("Early stopping triggered")
        break

Staring epoch 0


/home/alkelly2/ece556/myvenv/lib/python3.13/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch:  0 / 10  Loss:  1.6245702958121933  Accuracy: 62.330827067669176
Staring epoch 1
Epoch:  1 / 10  Loss:  0.5387591684895351  Accuracy: 87.21804511278195
Staring epoch 2
Epoch:  2 / 10  Loss:  0.2796334642590138  Accuracy: 93.40852130325814
Staring epoch 3
Epoch:  3 / 10  Loss:  0.1791562887940341  Accuracy: 96.09022556390977
Staring epoch 4
Epoch:  4 / 10  Loss:  0.11535022501369242  Accuracy: 97.91979949874687
Staring epoch 5
Epoch:  5 / 10  Loss:  0.09507011887232276  Accuracy: 98.1203007518797
Staring epoch 6
Epoch:  6 / 10  Loss:  0.07551121261475005  Accuracy: 98.37092731829574
Staring epoch 7
Epoch:  7 / 10  Loss:  0.07842102327750515  Accuracy: 98.37092731829574
Staring epoch 8
Epoch:  8 / 10  Loss:  0.08851191976088983  Accuracy: 98.0701754385965
Staring epoch 9
Epoch:  9 / 10  Loss:  0.05994081554612271  Accuracy: 98.7218045112782


In [18]:
#Save trained model
PATH = './produce_net.pth'
torch.save(model, PATH)

In [19]:
#Load model
PATH = './produce_net.pth'
model = torch.load(PATH, weights_only=False)

In [20]:
#Find network validation accuracy
correct = 0
total = 0

model.eval() #enable evaluation mode

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images) #get predicted classes
        
        #compare predicted and actual labels
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        #get network accuracy
        val_acc = 100 * (correct / total)
        
    print(f"Validation accuracy of the network: {val_acc:.2f} %")

Validation accuracy of the network: 9.7e+01 %


In [31]:
#Get model information
model_size = get_model_size(model)
print(f"Model Size {model_size/MiB:.2f} MiB")

num_parameters = get_num_parameters(model)
print(f"Parameter Count (M) {num_parameters/1e6:.2f}")

dummy_input = torch.randn(1, 3, 224, 224).to(device)
macs = get_model_macs(model, dummy_input)
print(f"MACS (B) {macs/1e9:.2f}")

Model Size 42.73 MiB
Parameter Count (M) 11.20
MACS (B) 1.82
